# step08 - L3 gridding: calibrated, flagged products onto the 1 m down-cast bins

The last step of the level scheme agreed with Amala and Edem (docs/DATA_LEVELS.md): **L2** holds the merged,
calibrated, flagged products at native resolution; **L3** is those products averaged onto 1 m bins. This
notebook writes one table per cast to `L3/gridded/<cast>_L3_1m_down.csv`.

## What a row is

Each row is one of the bins Sea-Bird **Bin Average** wrote for that cast (`<cast>_1m_down.cnv`: pressure bins
of 1 dbar, down-cast only, loop-edit-flagged scans excluded). The CTD columns are SBE's own bin averages,
carried unchanged; the nitrate and chlorophyll columns are averaged onto the same bins in the same way, so a
row describes one parcel of water with every variable computed alike.

* **Vertical coordinate: pressure.** The SBE bins are pressure bins (`binavg_bintype = decibars`), so the L2
  values are binned on `pressure_db`, not depth (at 5 N depth is 0.56% smaller than pressure; mixing the two
  would misplace values by up to a bin at 180 dbar). `depSM` is carried for the depth of each bin.
* **Bin assignment.** Bin b spans [b - 0.5, b + 0.5) dbar around SBE's bin centre b. A value exactly on an
  edge (x.500 dbar, about 1 frame in 1000) goes up; Sea-Bird does not document its edge rule, so such a frame
  can sit one bin away from where Bin Average would put it.
* **Down-cast only.** Frames at or before the time of the CTD's maximum pressure (step03's `leg`). On this
  SUNA the down- and up-cast disagree across the nitracline because the flow cell lags (down reads low, up
  reads high); averaging both legs would make each bin's meaning depend on how many frames each leg
  contributed, and the up-cast is dominated by bottle stops.
* **Only where the CTD's own scan was used.** Loop Edit flags scans while the package is held still or reverses
  (including a heaving bottom stop, where every scan below the deepest point so far counts as a reversal), and
  Bin Average leaves them out of the CTD bins. step03 gives each SUNA frame the 4 Hz Derived-file scan taken at
  the same moment; a frame is used only if Loop Edit kept that scan, and it is binned by that scan's pressure.
  So a nitrate frame enters a bin exactly when the CTD's scan did. On P45_06 CTD_04 this keeps about 11 frames
  in the 93 dbar bin (the CTD's 44 scans at 4 Hz) instead of 91 from a 100 s bottom stop. Casts without a
  Derived file fall back to the 1 s record's gaps, with a note.
* **Rows keyed by bin.** `pressure_db` is the bin centre. Bin Average wrote the centre on P45_06 and the mean
  pressure of the bin's scans on P45_07; that value is kept as `ctd_pressure_db`, and a file whose pressures do
  not map one-to-one onto bins is refused.

## The statistics

* **Value = arithmetic mean** of the frames in the bin, the statistic Bin Average uses for the CTD columns.
* **n = frames averaged**, in every row (`nitrate_n`, `chl_n`; `ctd_n_scans` for the CTD). The SUNA samples at
  about 1 Hz while the package descends at roughly 0.5 to 1 m/s, so most nitrate bins hold 1 or 2 frames: a
  bin value is then essentially a single measurement, and its uncertainty is the sensor's, not a reduced
  standard error. Bins with no frame are left empty, never interpolated.
* **sd = sample standard deviation** (ddof 1) where n >= 2. With n of 2 or 3 it indicates the spread between
  frames, not a reliable estimate of it.
* **Minimum frames per bin: 1**, as Bin Average's `MinScansPerBin`.

## What travels with the values

* `nitrate_lag_limited`: step03's flag. The calibrated SUNA nitrate is correct in homogeneous water but smears
  the nitracline; the bottles remain the nitrate structure of record.
* `suna_time_offset_s`: the SUNA clock correction step02 applied (0 when none).
* `chl_npq_risk`: True in bins at or above 10 dbar on casts started in daylight (07 to 17 h local solar time, UTC + longitude/15, from the cast summary;
  clock time UTC + `utc_offset_hours` for a cast with no position), the
  criterion step07 uses to drop quenched bottles from its fit. Fluorescence there is light-suppressed and the
  calibrated chlorophyll reads low.
* Only nitrate frames that passed SUNA spectral QC (`L1_clean`) are used. A cast whose frames all failed
  (P45_06 CTD_07: the long-wave baseline test) has no nitrate here, by design.

**Chlorophyll today** comes from step07's product, which is still computed on the 1 m bins, so `chl_n` is 1.
When step07 moves to native resolution (phase 4, next), the same code averages its scans per bin, using its
`leg` and loop-edit `flag` columns if present.


In [ ]:
from pathlib import Path
import sys
import pandas as pd

try:
    _anchor = Path(__file__).resolve()
except NameError:
    _anchor = Path.cwd()
for _p in (_anchor, *_anchor.parents):
    if (_p / "project_paths.py").exists():
        sys.path.insert(0, str(_p)); break
from project_paths import project_root, product_dir, cruise_dir
try:
    from suncat.gridding import GridConfig, grid_cruise
    from suncat.casts import read_cast_positions
except ImportError:                                   # suncat not installed: use the copy in the repo
    sys.path.insert(0, str(Path(_p) / "src"))
    from suncat.gridding import GridConfig, grid_cruise
    from suncat.casts import read_cast_positions

# ------------------------------------------------------------------ settings
CRUISE_ID   = "P45_07"
CHL_PRODUCT = "chl_calibrated"        # NPQ points removed from the fit (step07); "chl_calibrated_noNPQ" = kept
# NPQ daylight: local solar time (UTC + longitude/15) at the cast start, from the cast summary; utc_offset_hours
# is only the fallback for a cast with no position.
CFG = GridConfig(npq_depth_dbar=10.0, day_start_hour=7, day_end_hour=17, solar_time=True, utc_offset_hours=0.0,
                 min_frames=1)

CTD_DIR   = product_dir(CRUISE_ID, "ctd_grid_1m_down")
NITRATE   = product_dir(CRUISE_ID, "nitrate_qc")
CHL_FILE  = product_dir(CRUISE_ID, CHL_PRODUCT) / f"{CRUISE_ID}__chl_L3_corrected.csv"
SUNA_L2   = product_dir(CRUISE_ID, "suna_l2")
OUT_DIR   = product_dir(CRUISE_ID, "grid_1m_down", create=True)
AUDIT_DIR = cruise_dir(CRUISE_ID) / "_audit" / "gridding"
AUDIT_DIR.mkdir(parents=True, exist_ok=True)
print("cruise", CRUISE_ID, "| CTD bins:", CTD_DIR, "| chl:", CHL_FILE.name, "(exists:", CHL_FILE.exists(), ")")
print("out:", OUT_DIR)


In [ ]:
_casts = [f.name[: -len("_1m_down.cnv")] for f in sorted(CTD_DIR.glob("*_1m_down.cnv"))]
POSITIONS = read_cast_positions(cruise_dir(CRUISE_ID) / "metadata", CRUISE_ID, _casts)
_nsun = int(POSITIONS["lon"].notna().sum()) if len(POSITIONS) else 0
print(f"NPQ daylight: solar time for {_nsun} of {len(_casts)} casts"
      + (f" ({POSITIONS['source'].iloc[0]})" if len(POSITIONS) else "")
      + (f"; the others use clock time UTC{CFG.utc_offset_hours:+g} h" if _nsun < len(_casts) else ""))
tables, stats = grid_cruise(CRUISE_ID, CTD_DIR, NITRATE, CHL_FILE, SUNA_L2, OUT_DIR, CFG, positions=POSITIONS)
stats.to_csv(AUDIT_DIR / f"{CRUISE_ID}__L3_gridding_summary.csv", index=False, encoding="utf-8-sig")
with pd.option_context("display.width", 220, "display.max_columns", 30):
    print(stats.to_string(index=False))
print(f"\n{len(tables)} casts gridded -> {OUT_DIR}")
print("summary ->", AUDIT_DIR / f"{CRUISE_ID}__L3_gridding_summary.csv")


## Quick look

One cast at a time: the gridded nitrate and chlorophyll against pressure, with the number of frames in each
bin, so thin or empty bins are seen next to the values. Saved to `_audit/gridding/`.


In [ ]:
import matplotlib
try:
    get_ipython().run_line_magic("matplotlib", "inline")
except Exception:
    matplotlib.use("Agg")
import matplotlib.pyplot as plt

CAST = next(iter(tables), None)       # set to a cast id to look at another one

if CAST is not None:
    t = tables[CAST]
    fig, ax = plt.subplots(1, 3, figsize=(11, 5.5), sharey=True)
    ax[0].plot(t["nitrate_uM"], t["pressure_db"], "o-", ms=3, color="#2E6FB0")
    ax[0].set_xlabel("nitrate (uM, calibrated, lag-limited)")
    ax[1].plot(t["chl_mg_m3"], t["pressure_db"], "o-", ms=3, color="#1a9850")
    q = t["chl_npq_risk"].fillna(False).astype(bool)
    ax[1].plot(t.loc[q, "chl_mg_m3"], t.loc[q, "pressure_db"], "o", ms=6, mfc="none", mec="#d62728",
               label="NPQ risk")
    ax[1].set_xlabel("Chl-a (mg/m3, fluorescence-derived)"); ax[1].legend(fontsize=8, frameon=False)
    ax[2].barh(t["pressure_db"], t["nitrate_n"], height=0.8, color="#2E6FB0", alpha=0.6, label="nitrate n")
    ax[2].set_xlabel("frames per bin"); ax[2].legend(fontsize=8, frameon=False)
    ax[0].set_ylabel("pressure (dbar)"); ax[0].invert_yaxis()
    for a in ax:
        for s in ("top", "right"): a.spines[s].set_visible(False)
    fig.suptitle(f"{CAST}  L3 1 m down-cast bins", fontsize=11)
    fig.tight_layout()
    out = AUDIT_DIR / f"{CAST}__L3_quicklook.png"
    fig.savefig(out, dpi=130); plt.show()
    print("saved", out)
